# Databricks Connect Demo

This notebook demonstrates how to use Databricks Connect to run Spark code locally against a Databricks workspace.

**Configuration:**
- Profile: `DEFAULT`
- Workspace: e2-demo-field-eng.cloud.databricks.com
- Compute: Serverless

## 1. Connect to Databricks

In [2]:
from databricks.connect import DatabricksSession

# Create a Spark session connected to Databricks
spark = DatabricksSession.builder.profile("DEFAULT").getOrCreate()

print(f"Connected to Databricks!")
print(f"Spark version: {spark.version}")

Connected to Databricks!
Spark version: 4.1.0


## 2. List Available Catalogs

In [3]:
# List all catalogs in Unity Catalog
catalogs = spark.sql("SHOW CATALOGS")
catalogs.show(truncate=False)

+-------------------------------------------------------------------------------+
|catalog                                                                        |
+-------------------------------------------------------------------------------+
|00vsdb                                                                         |
|00vsdb2                                                                        |
|00vsdb3                                                                        |
|03-scp-glue                                                                    |
|10_16_share                                                                    |
|118553_catalog                                                                 |
|118_group_sample_uk_business_data_full_uk_b2b_coverage_marketable_or_analytical|
|202509_ski_poc                                                                 |
|2026_2_3_demo_nobunagasambition                                                |
|365scores      

## 3. Explore a Catalog

In [5]:
# Change this to a catalog you have access to
CATALOG = "sv_aibuilder_saas_demo"

# List schemas in the catalog
schemas = spark.sql(f"SHOW SCHEMAS IN {CATALOG}")
schemas.show(truncate=False)

+------------------+
|databaseName      |
+------------------+
|bronze            |
|default           |
|gold              |
|information_schema|
|saas_sandbox      |
|silver            |
+------------------+



In [6]:
# List tables in a schema
SCHEMA = "gold"

tables = spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}")
tables.show(truncate=False)

+--------+-------------------------+-----------+
|database|tableName                |isTemporary|
+--------+-------------------------+-----------+
|gold    |agg_daily_product_metrics|false      |
|gold    |agg_user_lifetime_summary|false      |
|gold    |dim_user                 |false      |
|gold    |fact_event               |false      |
|gold    |fact_order               |false      |
+--------+-------------------------+-----------+



## 4. Query Sample Data

In [ ]:
# Query the NYC taxi trips table
df = spark.sql("""
    SELECT 
        pickup_datetime,
        dropoff_datetime,
        trip_distance,
        fare_amount,
        tip_amount
    FROM samples.nyctaxi.trips
    LIMIT 10
""")

df.show()

## 5. DataFrame Operations

In [ ]:
from pyspark.sql import functions as F

# Read the full table as a DataFrame
trips_df = spark.table("samples.nyctaxi.trips")

# Show schema
trips_df.printSchema()

In [ ]:
# Aggregate statistics
stats = trips_df.agg(
    F.count("*").alias("total_trips"),
    F.round(F.avg("trip_distance"), 2).alias("avg_distance_miles"),
    F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
    F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
    F.round(F.sum("fare_amount"), 2).alias("total_revenue")
)

stats.show()

In [ ]:
# Group by pickup hour
hourly_trips = (
    trips_df
    .withColumn("pickup_hour", F.hour("pickup_datetime"))
    .groupBy("pickup_hour")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare")
    )
    .orderBy("pickup_hour")
)

hourly_trips.show(24)

## 6. Convert to Pandas

In [ ]:
# Convert Spark DataFrame to Pandas for local analysis
hourly_pandas = hourly_trips.toPandas()
hourly_pandas.head(10)

In [ ]:
# Plot with pandas (if matplotlib is installed)
try:
    import matplotlib.pyplot as plt
    
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(hourly_pandas["pickup_hour"], hourly_pandas["trip_count"])
    ax.set_xlabel("Hour of Day")
    ax.set_ylabel("Number of Trips")
    ax.set_title("NYC Taxi Trips by Hour")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Install matplotlib for plotting: uv pip install matplotlib")

## 7. Write Data (Example)

In [ ]:
# Example: Create a local DataFrame and write to Databricks
# Uncomment and modify catalog/schema to your own

# from datetime import datetime
# 
# data = [
#     (1, "Alice", datetime.now()),
#     (2, "Bob", datetime.now()),
#     (3, "Charlie", datetime.now())
# ]
# 
# df = spark.createDataFrame(data, ["id", "name", "created_at"])
# 
# df.write.mode("overwrite").saveAsTable("my_catalog.my_schema.test_table")

## 8. Clean Up

In [ ]:
# Stop the Spark session when done
# spark.stop()
# print("Spark session stopped.")

## 9. SaaS Monthly Sales Analysis

Analyzing sales trends from `sv_aibuilder_saas_demo.gold.fact_order` for the last 3 months.

In [7]:
# Monthly Sales Trend - Last 3 Months (Paying & Free Tier Users)
monthly_sales = spark.sql("""
    SELECT 
        DATE_FORMAT(order_month, 'yyyy-MM') AS month,
        COUNT(DISTINCT user_id) AS active_users,
        COUNT(*) AS total_orders,
        SUM(CASE WHEN is_revenue THEN order_value ELSE 0 END) AS revenue,
        ROUND(SUM(CASE WHEN is_revenue THEN order_value ELSE 0 END) / COUNT(DISTINCT user_id), 2) AS arpu
    FROM sv_aibuilder_saas_demo.gold.fact_order
    WHERE order_month >= ADD_MONTHS(DATE_TRUNC('month', CURRENT_DATE()), -3)
    GROUP BY order_month
    ORDER BY order_month DESC
""")

print("📊 Monthly Sales Trend (Last 3 Months)")
monthly_sales.show(truncate=False)

# Month-over-Month Growth
print("\n📈 Month-over-Month Changes")
spark.sql("""
    WITH monthly AS (
        SELECT 
            DATE_FORMAT(order_month, 'yyyy-MM') AS month,
            SUM(CASE WHEN is_revenue THEN order_value ELSE 0 END) AS revenue,
            COUNT(DISTINCT user_id) AS users
        FROM sv_aibuilder_saas_demo.gold.fact_order
        WHERE order_month >= ADD_MONTHS(DATE_TRUNC('month', CURRENT_DATE()), -3)
        GROUP BY order_month
        ORDER BY order_month
    )
    SELECT 
        month,
        revenue,
        users,
        ROUND(100 * (revenue - LAG(revenue) OVER (ORDER BY month)) / LAG(revenue) OVER (ORDER BY month), 1) AS revenue_growth_pct,
        ROUND(100 * (users - LAG(users) OVER (ORDER BY month)) / LAG(users) OVER (ORDER BY month), 1) AS user_growth_pct
    FROM monthly
""").show(truncate=False)

📊 Monthly Sales Trend (Last 3 Months)
+-------+------------+------------+----------+------+
|month  |active_users|total_orders|revenue   |arpu  |
+-------+------------+------------+----------+------+
|2026-02|5188        |7383        |2894496.61|557.92|
|2026-01|6359        |9965        |3822492.80|601.12|
|2025-12|6390        |10051       |3795013.58|593.90|
|2025-11|2296        |2601        |1000709.56|435.85|
+-------+------------+------------+----------+------+


📈 Month-over-Month Changes
+-------+----------+-----+------------------+---------------+
|month  |revenue   |users|revenue_growth_pct|user_growth_pct|
+-------+----------+-----+------------------+---------------+
|2025-11|1000709.56|2296 |NULL              |NULL           |
|2025-12|3795013.58|6390 |279.2             |178.3          |
|2026-01|3822492.80|6359 |0.7               |-0.5           |
|2026-02|2894496.61|5188 |-24.3             |-18.4          |
+-------+----------+-----+------------------+---------------+



## 10. Monthly Renewal Rate

**Calculation:** Users with revenue orders in both Month N-1 and Month N / Users in Month N-1

In [ ]:
# Monthly Renewal Rate - Self-join to compare consecutive months
renewal_rate = spark.sql("""
    WITH monthly_users AS (
        SELECT DISTINCT 
            DATE_FORMAT(order_month, 'yyyy-MM') AS month,
            user_id
        FROM sv_aibuilder_saas_demo.gold.fact_order
        WHERE is_revenue = true
    ),
    retention AS (
        SELECT 
            curr.month,
            COUNT(DISTINCT curr.user_id) AS current_users,
            COUNT(DISTINCT prev.user_id) AS renewed_users,
            COUNT(DISTINCT CASE WHEN prev.user_id IS NULL THEN curr.user_id END) AS new_users
        FROM monthly_users curr
        LEFT JOIN monthly_users prev 
            ON curr.user_id = prev.user_id 
            AND prev.month = DATE_FORMAT(ADD_MONTHS(TO_DATE(CONCAT(curr.month, '-01')), -1), 'yyyy-MM')
        WHERE curr.month >= '2025-12'
        GROUP BY curr.month
    )
    SELECT 
        month,
        current_users,
        renewed_users,
        new_users,
        ROUND(100.0 * renewed_users / LAG(current_users) OVER (ORDER BY month), 1) AS renewal_rate_pct
    FROM retention
    ORDER BY month
""")

print("📊 Monthly Renewal Rate")
renewal_rate.show(truncate=False)